# CSV handling, cookbook creation and meal recommendation

### Funkcjonalności:

- możesz do csv dopisać jakiś przepis nowy jak znajdziesz, plus żeby w ogóle link do czegoś takiego istniał
- opcja usuwania przepisu z bazy, jeśli Ci się nie podoba 
- tworzy 'cookbook' - czyli listę wszystkich instancji klasy przepis
- rekomenduje Ci danie na postawie metody najmniejszych kwadratów (z przydzielonymi wagami)

# Importy bibiliotek i zmienne globalne

In [232]:
import csv
import requests
import os
from bs4 import BeautifulSoup

In [233]:
cookbook = []

# Walidacja inputów

In [234]:
def validate_number(message: str) -> int:
    """
    Gets the numeric input and checks if it's correct, so if it's an integer and if it's higher than zero.
    
    Args:
        message: message that the user gets while asked for input
    
    Returns:
        int:  validated value given by the user 
    """
    while True:
        try:
            value = int(input(message).strip())
            if value <= 0:
                print("Błąd: Wartość musi być większa od zera.")
                continue
            return value
        
        except ValueError:
            print("Błąd: Podana wartość musi być liczbą.")

In [235]:
def validate_selection(message: str, possible_options: list[str]) -> str:
    """
    Gets the user's chosen option and checks if it's correct, so if it's in the list of possible options.
    
    Args:
        message: message that the user gets while asked for input

    Returns:
        str:  validated option given by the user 
    """
    while True:
        wybor = input(f"{message} ({'/'.join(possible_options)}): ").strip().lower()
        if wybor in possible_options:
            return wybor
        
        print(f"Błąd: Wybierz jedną z opcji: {'/'.join(possible_options)}")

In [236]:
def get_users_input() -> tuple:
    """
    Gets and validates all of the user's inputs needed for all the program's calculations.
    
    Args:

    Returns:
        tuple:  all of the user's inputs
    """
    sex = validate_selection("Podaj swoją płeć", ['k', 'm'])
    age = validate_number("Podaj swój wiek: ")
    weight = validate_number("Podaj swoją wagę (kg): ")
    height = validate_number("Podaj swój wzrost (cm): ")

    # Connects options to numeric values, so it gets easier for the user to type in their activity level
    act_levels = {
        '1': "Brak ćwiczeń", 
        '2': "Lekka aktywność (1-3 dni)", 
        '3': "Umiarkowana (3-5 dni)", 
        '4': "Duża (6-7 dni)"
    }
    
    print("\nPoziomy aktywności: 1. Brak ćwiczeń, 2. Lekka aktywność (1-3 dni), 3. Umiarkowana (3-5 dni), 4. Duża (6-7 dni)")
    act_index = validate_selection("Wybierz poziom aktywności (numer)", ['1', '2', '3', '4'])
    phys_act = act_levels[act_index]
    diet_type = validate_selection("Podaj typ diety", ['bezglutenowa', 'wegańska', 'wegetariańska', 'standardowa'])
    goal = validate_selection("Podaj swój cel", ['przytyć', 'schudnąć', 'utrzymać wagę'])
    
    return sex, age, weight, height, phys_act, diet_type, goal

# Pobieranie przepisów ze strony w jedną 'książkę kucharską; :)

In [237]:
class recipe:
    def __init__(self, title: str, macro: dict, cooking_time: float, number_of_portions: str, diet: list[str], ingredients: list[str]):
        self.title = title
        self.macro = macro
        self.cooking_time = cooking_time
        self.number_of_portions = str(number_of_portions)
        self.diet = diet
        self.ingredients = ingredients
        cookbook.append(self) if self not in cookbook else None

    def __repr__(self) -> str:
        return f"{self.title}:\nMacro: {self.macro}\nCzas gotowania: {self.cooking_time} minut\nLiczba porcji: {self.number_of_portions}\nDieta: {', '.join(self.diet)}\nSkładniki: {', '.join(self.ingredients)}"
    
    # Returns True if both class objects have the exact same attributes and values
    def __eq__(self, other) : 
        return self.__dict__ == other.__dict__

In [238]:
def create_soup(recipe_name: str) -> BeautifulSoup:
    """
    Sends an HTTP GET request to the aniagotuje urls and gets the site's content

    Args:
        recipe_name: name of the recipe that program appends to the base url

    Returns:
        BeautifulSoup: an object containing the parsed HTML of the recipe page
    """
    aniagotuje_url = "https://aniagotuje.pl/przepis/"
    response = requests.get(aniagotuje_url + recipe_name)
    response.raise_for_status()
        
    return BeautifulSoup(response.content, 'html.parser')

In [239]:
def get_recipe_info(soup: BeautifulSoup) -> tuple | str:
    """
    Parses the recipe page to extract portions, diet type, cooking time, and macros

    Args:
        soup: the BeautifulSoup object representing the parsed HTML of the recipe page

    Returns:
        tuple: collection of extracted data - portions (str), diet (list), cooking_time (int), macros (dict)
        str: message about an error
    """
    # Locates the main recipe info container in site's content
    recipe_info = soup.find('p', class_='recipe-info')
    if not recipe_info:
        return "Nie znaleziono informacji o przepisie"

    # Extracts clean text from recipe info container
    full_text = recipe_info.get_text(" ", strip=True)
    
    # Initializes default results structure
    results = {
        "portions": "Nie podano",
        "diet": [],
        "cooking_time": 0,
        "macros": {}
    }

    # Extracts cooking time
    if "Czas przygotowania:" in full_text:
        # Cuts out whats after "Czas przygotowania:" and before "Liczba porcji:"
        time_raw = full_text.split("Czas przygotowania:")[1].split("Liczba porcji:")[0].strip()
        parts = time_raw.split()
        # If there's a numeric value
        if parts[0].isdigit():
            val = int(parts[0])
            # If it's about hours ("godzina"/"godziny/"godzin"), it turns this value into minutes (x * 600)
            if "godz" in time_raw:
                results["cooking_time"] = val * 60
            else:
                results["cooking_time"] = val
    
    # Extracts portions
    if "Liczba porcji:" in full_text:
        # Cuts out whats after "Liczba porcji:" and before "W 100"
        results["portions"] = full_text.split("Liczba porcji:")[1].split("W 100")[0].strip()

    # Extracts diet type
    if "Dieta:" in full_text:
        # Cuts out whats after "Dieta:"
        diet_val = full_text.split("Dieta:")[1].strip()
        if diet_val:
            # Splits string into list elements (e.g. 'bezglutenowa, wegańska' -> ['bezglutenowa', 'wegańska'])
            results["diet"] = [d.strip().lower() for d in diet_val.split(',') if d.strip()]
        else:
            # Sets diet to standard value, if there's nothing after "Diet:"
            results["diet"] = ["standardowa"]
    else:
        # Sets diet to standard value, if there's no "Diet:" info on the site
        results["diet"] = ["standardowa"]

    # Macronutrients (structured approach using itemprop attributes)
    macro_map = {
        'calories': 'Kalorie (kcal)',
        'carbohydrateContent': 'Węglowodany (g)',
        'sugarContent': 'Cukry (g)',
        'proteinContent': 'Białko (g)',
        'fatContent': 'Tłuszcze (g)'
    }

    for item_prop, label in macro_map.items():
        # Find specific meta-tags or spans based on Schema.org microdata
        element = soup.find(attrs={"itemprop": item_prop})
        if element:
            raw_text = element.get_text(strip=True) 
            # Split to separate the number from the unit
            parts = raw_text.split(" ")

            if len(parts) > 0:
                # aniagotuje site uses commas for decimals, so the program converts it to dots for float compatibility
                first_part = parts[0].replace(",", ".")

                try:
                    value = float(first_part)
                    results["macros"][label] = value
                except ValueError:
                    # Sets dict value to 0 if there's an error in data parsing
                    results["macros"][label] = 0

    return results["portions"], results["diet"], results["cooking_time"], results["macros"]

In [240]:
def get_recipe_ingredients(soup: BeautifulSoup) -> list | None:
    """
    Parses the recipe page to extract ingredients

    Args:
        soup: the BeautifulSoup object representing the parsed HTML of the recipe page

    Returns:
        list: extracted ingredients data 
        str: message about an error
    """
    items = soup.find_all(attrs={"itemprop": "recipeIngredient"})
    if not items:
        return "Nie znaleziono informacji o składnikach przepisu"
    
    # Processes each found element into a clean list element
    ingredients_list = [item.get_text(" ", strip=True) for item in items]
    
    return ingredients_list

In [241]:
def get_recipe_title(soup: BeautifulSoup) -> str | None:
    """
    Parses the recipe page to extract title

    Args:
        soup: the BeautifulSoup object representing the parsed HTML of the recipe page

    Returns:
        str: extracted title data or a message about an error
    """
    # Searches for an h1 tag with Schema.org 'itemprop="name"' attribute
    title_tag = soup.find('h1', attrs={'itemprop': 'name'})
    
    # If title_tag is None, searches for a generic h1 tag
    if not title_tag:
        title_tag = soup.find('h1')
        # If title_tag is still None, returns an error message
        if not title_tag:
            return "Nie znaleziono informacji o tytule przepisu"
        
    return title_tag.get_text(strip=True)

In [242]:
def get_recipe(recipe_name: str) -> recipe:
    """
    Combines the scraping process to create a new recipe object
    
    Args:
        recipe_name: name of the recipe used to build the target url

    Returns:
        recipe: class object for the given recipe
    """
    soup = create_soup(recipe_name) 

    number_of_portions, diet, cooking_time, macro = get_recipe_info(soup)
    ingredients = get_recipe_ingredients(soup)
    title = get_recipe_title(soup)
    
    return recipe(title, macro, cooking_time, number_of_portions, diet, ingredients)

# Rekomendacja top dania najbardziej odpowiadającego potrzebom żywieniowym i typowi diety

In [243]:
def find_best_match(cookbook: list, goals: dict, target_diet: str = 'standardowa') -> recipe | None:
    """
    Finds the most suitable recipe based on nutritional goals and dietary preferences. 
    
    It uses a weighted sum of squared differences to calculate a 'distance' score. 
    The recipe with the lowest score is returned
    
    Args:
        cookbook: list of recipe objects
        goals: dictionary containing target values for calories, carbs, sugar, protein, and fat
        target_diet: diet type to filter by -'standardowa' by default

    Returns:
        recipe: class object that best matches the provided goals
        None: if no match is found
    """
    best_recipe = None
    # Initializes with infinity so any first valid recipe will have a lower score
    min_score = float('inf')

    # Links internal goal keys to the keys used in the recipe macro dictionary (like {macro_goal : self.macro[smth]})
    macro_map = {
        'calories': 'Kalorie (kcal)',
        'carbs': 'Węglowodany (g)',
        'sugar': 'Cukry (g)',
        'protein': 'Białko (g)',
        'fat': 'Tłuszcze (g)'
    }

    # Normalize the scale differences with weights (calories are hundreds, macros are tens)
    weights = {
        'calories': 0.1,
        'carbs': 1.0,
        'sugar': 1.0,
        'protein': 1.0,
        'fat': 1.0
    }

    for recipe in cookbook:
        # Checks if the user has diet type other than 'standardowa'
        if target_diet != 'standardowa':
            # If so, then checks if the user's diet type is in recipe description. 
            # If not, then it goes to the next recipe
            if target_diet not in recipe.diet:
                continue
                
        score = 0
        for goal_key, macro_key in macro_map.items():
            # Gets values as floats for mathematical operations
            val = float(recipe.macro.get(macro_key, 0))
            goal = float(goals[goal_key])
            
            # Weighted Least Squares formula: (goal - actual) * weight, then squared
            diff = (goal - val) * weights[goal_key]
            score += diff**2

        # If this recipe's total score is lower than the current minimum, it becomes the new best match
        if score < min_score:
            min_score = score
            best_recipe = recipe

    return best_recipe

# Procesowanie pliku csv

In [244]:
def load_recipes_from_file(filename="recipes.csv") -> list:
    """
    Loads recipe list from the CSV file. If the file doesn't exist, returns an empty list.
    
    Args:
        filename: name of the file with recipes, by default it should be 'recipes.csv' (pre-made recipe list)

    Returns:
        list:  list of all the recipes that the file contains
    """
    recipes = []
    
    # Checks if the path to a file exists, if not - then returns an empty list
    if not os.path.exists(filename):
        return recipes

    try:
        with open(filename, "r", encoding="utf-8") as f:
            reader = csv.reader(f)
            for row in reader:
                # Checks if a line in the file is not empty
                if row:
                    # Gets the first value in the row and adds it to the list
                    recipes.append(row[0].strip())
    except Exception as e:
        print(f"Nieoczekiwany błąd podczas wczytywania pliku: {e}")
    
    return recipes

In [245]:
def add_recipe_to_file(recipe_url_name: str, filename='recipes.csv') -> None:
    """
    Adds recipe to the CSV file. If the file doesn't exist, returns an empty list.
    
    Args:
        recipe_url_name: name of the recipe taken from aniagotuje url, that user tries to add to the file
        filename: name of the file with recipes, by default it should be 'recipes.csv' (pre-made recipe list)

    Returns:
        None: function returns a None value, if the recipe already is in the file, or prints out a success 
                message or an error message
    """
    recipe_url_name = recipe_url_name.strip().lower()
    
    # Checks if the user's recipe already exists in the csv file
    if recipe_url_name in load_recipes_from_file(filename):
        print(f"Wpis '{recipe_url_name}' już istnieje w pliku.")
        return
            
    try:
        # If create_soup() raises an error, recipe won't be added to the csv file
        soup = create_soup(recipe_url_name)
        title = get_recipe_title(soup)
        
        # Appends recipe name to the csv file
        with open(filename, 'a', encoding='utf-8', newline='') as f:
            csv.writer(f).writerow([recipe_url_name])
        
        print(f"Dodano przepis {title}.")

    # Handles the errors raised by create_soup()
    except requests.exceptions.HTTPError:
        print("Błąd: Strona dla Twojego przepisu nie istnieje.")
    except requests.exceptions.ConnectionError:
        print("Błąd: Problem z połączeniem internetowym.")
    except Exception as e:
        print(f"Nieoczeliwany błąd: {e}")

In [246]:
def remove_recipe_from_file(recipe_url_name: str, filename='recipes.csv') -> None:
    """
    Removes recipe from the CSV file. If the file doesn't exist, returns an empty list.
    
    Args:
        recipe_url_name: name of the recipe taken from aniagotuje url, that user tries to remove from the file
        filename: name of the file with recipes, by default it should be 'recipes.csv' (pre-made recipe list)

    Returns:
        None: function doesn't return a value, but it prints out a success message or an error message
    """
    recipe_url_name = recipe_url_name.strip().lower()
    recipes = load_recipes_from_file(filename)
    
    if recipe_url_name in recipes:
        soup = create_soup(recipe_url_name)
        title = get_recipe_title(soup)
        
        # Removes the user's recipe from the recipe list
        recipes.remove(recipe_url_name) 
        # Clears the file and writes down modificated recipe list (there's no an easier way to do this)
        with open(filename, "w", encoding="utf-8", newline='') as f:
            writer = csv.writer(f)
            for recipe in recipes:
                writer.writerow([recipe])
            
        print(f"Usunięto {title}")
    else:
        print(f"Nie znaleziono {title}")

# Example code that could be in main.py

In [247]:
recipies_base = load_recipes_from_file("recipes.csv")

for recipe_name in recipies_base:
    get_recipe(recipe_name)

In [248]:
cookbook

[Tatar ze śledzia z suszonymi pomidorami:
 Macro: {'Kalorie (kcal)': 252.0, 'Węglowodany (g)': 8.0, 'Cukry (g)': 5.0, 'Białko (g)': 10.0, 'Tłuszcze (g)': 20.0}
 Czas gotowania: 20 minut
 Liczba porcji: około 320 g
 Dieta: bezglutenowa
 Składniki: filety śledziowe a'la matjas z oleju 180 g - ok. 2 płaty, pomidory suszone z oleju 50 g, olej ze słoika z suszonymi pomidorami 1 łyżeczka, jabłko - najlepiej kruche, żółty miąższ 50 g, orzechy włoskie 20 g, mała cebulka szalotka 25 g, świeżo mielony pieprz spora szczypta, natka pietruszki do dekoracji,
 Karp z pieczarkami:
 Macro: {'Kalorie (kcal)': 123.0, 'Węglowodany (g)': 3.0, 'Cukry (g)': 1.0, 'Białko (g)': 12.0, 'Tłuszcze (g)': 7.0}
 Czas gotowania: 60 minut
 Liczba porcji: 4 mniejsze porcje
 Dieta: bezglutenowa
 Składniki: duży płat/filet z karpia 1 sztuka - 550 g, świeżo wyciśnięty sok z cytryny 2 łyżeczki, sól i pieprz po pół łyżeczki, pieczarki 250 g, cebula 180 g, olej do smażenia 3 łyżki, sól i pieprz po 1/3 łyżeczki,
 Placki twarog

In [249]:
# imagine it's the result of calculations in Zuzia's part
my_goals = {
    'calories': 250,
    'carbs': 10,
    'sugar': 7,
    'protein': 12,
    'fat': 25 
}

match = find_best_match(cookbook, my_goals)
print(f"Najlepszy dopasowany przepis to: {match.title}")
print(f"Jego makro: {match.macro}")
# maybe ask if the user wants to see the ingredients needed for this recipe?
# or let him skip this recipe and show the second best recipe? or top5?

Najlepszy dopasowany przepis to: Tatar ze śledzia z suszonymi pomidorami
Jego makro: {'Kalorie (kcal)': 252.0, 'Węglowodany (g)': 8.0, 'Cukry (g)': 5.0, 'Białko (g)': 10.0, 'Tłuszcze (g)': 20.0}


In [250]:
my_goals = {
    'calories': 250,
    'carbs': 10,
    'sugar': 7,
    'protein': 12,
    'fat': 25 
}

match = find_best_match(cookbook, my_goals, "wegańska")
print(f"Najlepszy dopasowany przepis to: {match.title}")
print(f"Jego makro: {match.macro}")

Najlepszy dopasowany przepis to: Brownie z dyni
Jego makro: {'Kalorie (kcal)': 183.0, 'Węglowodany (g)': 15.0, 'Cukry (g)': 9.0, 'Białko (g)': 6.0, 'Tłuszcze (g)': 11.0}
